In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Generate 1200 rows of data
n_rows = 1200
start_date = datetime(2026, 1, 1)

channels = ['Google Ads', 'google ads ', 'Facebook', 'FACEBOOK', 'LinkedIn', 'linkedin', 'Google Ads']
campaigns = ['Brand_Awareness_2026', 'Lead_Gen_Q1', 'Retargeting_Global', 'Promo_Spring']

data = {
    'Date': [
        (start_date + timedelta(days=random.randint(0, 150))).strftime('%Y-%m-%d') if i % 10 != 0
        else (start_date + timedelta(days=random.randint(0, 150))).strftime('%d/%m/%Y')
        for i in range(n_rows)
    ],
    'Channel': [random.choice(channels) for _ in range(n_rows)],
    'Campaign_Name': [random.choice(campaigns) for _ in range(n_rows)],
    'Impressions': np.random.randint(1000, 50000, size=n_rows),
    'Clicks': np.random.randint(10, 2500, size=n_rows),
    'Spend': np.round(np.random.uniform(50.0, 1500.0, size=n_rows), 2),
    'Leads': np.random.randint(0, 80, size=n_rows),
    'Revenue': np.round(np.random.uniform(100.0, 5000.0, size=n_rows), 2)
}

df = pd.DataFrame(data)

# Injecting artificial "messiness" for you to clean
# 1. Missing values
df.loc[df.sample(frac=0.04).index, 'Spend'] = np.nan
df.loc[df.sample(frac=0.03).index, 'Leads'] = np.nan

# 2. Impossible/Negative values (Glitches)
df.loc[df.sample(n=5).index, 'Spend'] = -150.00
df.loc[df.sample(n=3).index, 'Impressions'] = 9999999  # Outlier

# 3. Mismatched clicks/impressions (More clicks than impressions)
for idx in df.sample(n=10).index:
    df.at[idx, 'Clicks'] = df.at[idx, 'Impressions'] + 500

# Save to CSV
df.to_csv('raw_marketing_data.csv', index=False)
print("Success! 'raw_marketing_data.csv' has been generated with 1,200 rows.")

Success! 'raw_marketing_data.csv' has been generated with 1,200 rows.


In [ ]:
df.head()

,Date,Channel,Campaign_Name,Impressions,Clicks,Spend,Leads,Revenue
0,29/01/2026,Google Ads,Retargeting_Global,16795,1938,1186.57,50.0,2048.65
1,2026-01-07,google ads,Lead_Gen_Q1,1860,1023,564.52,79.0,4871.62
2,2026-03-12,Google Ads,Promo_Spring,39158,60,647.59,71.0,2766.57
3,2026-03-04,linkedin,Retargeting_Global,45732,1440,846.94,11.0,1448.67
4,2026-02-27,Google Ads,Retargeting_Global,12284,1802,1254.27,52.0,3576.25


In [ ]:
display(df)

,Date,Channel,Campaign_Name,Impressions,Clicks,Spend,Leads,Revenue
0,29/01/2026,Google Ads,Retargeting_Global,16795,1938,1186.57,50.0,2048.65
1,2026-01-07,google ads,Lead_Gen_Q1,1860,1023,564.52,79.0,4871.62
2,2026-03-12,Google Ads,Promo_Spring,39158,60,647.59,71.0,2766.57
3,2026-03-04,linkedin,Retargeting_Global,45732,1440,846.94,11.0,1448.67
4,2026-02-27,Google Ads,Retargeting_Global,12284,1802,1254.27,52.0,3576.25
...,...,...,...,...,...,...,...,...
1195,2026-03-12,google ads,Lead_Gen_Q1,39727,1684,152.79,46.0,825.00
1196,2026-02-07,LinkedIn,Lead_Gen_Q1,1126,445,503.65,47.0,4924.81
1197,2026-02-03,LinkedIn,Promo_Spring,28632,408,1406.18,41.0,1124.58
1198,2026-05-19,google ads,Retargeting_Global,42563,1244,1283.49,75.0,738.73


In [ ]:
import pandas as pd

# Load the messy dataset
df = pd.read_csv('raw_marketing_data.csv')

# Look at the first 10 rows
df.head(10)

# Check the data types and look for missing (Null) values
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           1200 non-null   object 
 1   Channel        1200 non-null   object 
 2   Campaign_Name  1200 non-null   object 
 3   Impressions    1200 non-null   int64  
 4   Clicks         1200 non-null   int64  
 5   Spend          1152 non-null   float64
 6   Leads          1164 non-null   float64
 7   Revenue        1200 non-null   float64
dtypes: float64(3), int64(2), object(3)
memory usage: 75.1+ KB


In [ ]:
# 1. Strip trailing/leading spaces and convert to lowercase
df['Channel'] = df['Channel'].str.strip().str.lower()

# 2. Check your work - this should print exactly THREE clean channels
print(df['Channel'].unique())

['google ads' 'linkedin' 'facebook']


In [ ]:
# Convert the Date column to a unified datetime format
df['Date'] = pd.to_datetime(df['Date'], format='mixed')

# Let's verify it worked by checking the data types again
print(df['Date'].dtype)
df.head()

datetime64[ns]


,Date,Channel,Campaign_Name,Impressions,Clicks,Spend,Leads,Revenue
0,2026-01-29,google ads,Retargeting_Global,16795,1938,1186.57,50.0,2048.65
1,2026-01-07,google ads,Lead_Gen_Q1,1860,1023,564.52,79.0,4871.62
2,2026-03-12,google ads,Promo_Spring,39158,60,647.59,71.0,2766.57
3,2026-03-04,linkedin,Retargeting_Global,45732,1440,846.94,11.0,1448.67
4,2026-02-27,google ads,Retargeting_Global,12284,1802,1254.27,52.0,3576.25


In [ ]:
# Fill missing values in Spend and Leads columns with 0
df['Spend'] = df['Spend'].fillna(0)
df['Leads'] = df['Leads'].fillna(0)

# Verify that there are no more missing values
print(df.isnull().sum())

Date             0
Channel          0
Campaign_Name    0
Impressions      0
Clicks           0
Spend            0
Leads            0
Revenue          0
dtype: int64


In [ ]:
# Keep only rows where spend is 0 or positive, impressions are reasonable, and clicks don't exceed impressions
cleaned_df = df[
    (df['Spend'] >= 0) &
    (df['Impressions'] < 5000000) &
    (df['Clicks'] <= df['Impressions'])
]

# See how many rows we have left (it should be just under 1,200)
print(f"Original rows: {len(df)}")
print(f"Cleaned rows: {len(cleaned_df)}")

Original rows: 1200
Cleaned rows: 1165


In [ ]:
cleaned_df.to_csv('clean_marketing_data.csv', index=False)


In [ ]:
import numpy as np

# 1. Calculate Click-Through Rate (as a decimal)
cleaned_df['CTR'] = np.where(cleaned_df['Impressions'] > 0, cleaned_df['Clicks'] / cleaned_df['Impressions'], 0)

# 2. Calculate Cost Per Click
cleaned_df['CPC'] = np.where(cleaned_df['Clicks'] > 0, cleaned_df['Spend'] / cleaned_df['Clicks'], 0)

# 3. Calculate Return on Investment
cleaned_df['ROI'] = np.where(cleaned_df['Spend'] > 0, (cleaned_df['Revenue'] - cleaned_df['Spend']) / cleaned_df['Spend'], 0)

# View the new columns
cleaned_df[['Channel', 'Impressions', 'Clicks', 'Spend', 'Revenue', 'CTR', 'CPC', 'ROI']].head()

/tmp/ipykernel_7443/3206884735.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df['CTR'] = np.where(cleaned_df['Impressions'] > 0, cleaned_df['Clicks'] / cleaned_df['Impressions'], 0)
/tmp/ipykernel_7443/3206884735.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df['CPC'] = np.where(cleaned_df['Clicks'] > 0, cleaned_df['Spend'] / cleaned_df['Clicks'], 0)
/tmp/ipykernel_7443/3206884735.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
T

,Channel,Impressions,Clicks,Spend,Revenue,CTR,CPC,ROI
0,google ads,16795,1938,1186.57,2048.65,0.115391,0.612265,0.726531
1,google ads,1860,1023,564.52,4871.62,0.550000,0.551828,7.629668
2,google ads,39158,60,647.59,2766.57,0.001532,10.793167,3.272101
3,linkedin,45732,1440,846.94,1448.67,0.031488,0.588153,0.710475
4,google ads,12284,1802,1254.27,3576.25,0.146695,0.696043,1.851260


In [ ]:
# Group by channel and aggregate our metrics
channel_summary = cleaned_df.groupby('Channel').agg(
    Total_Spend=('Spend', 'sum'),
    Total_Revenue=('Revenue', 'sum'),
    Total_Clicks=('Clicks', 'sum'),
    Total_Impressions=('Impressions', 'sum'),
    Total_Leads=('Leads', 'sum')
).reset_index()

# Recalculate overall metrics for the groups to ensure mathematical accuracy
channel_summary['Overall_CTR'] = channel_summary['Total_Clicks'] / channel_summary['Total_Impressions']
channel_summary['Overall_CPC'] = channel_summary['Total_Spend'] / channel_summary['Total_Clicks']
channel_summary['Cost_Per_Lead_CPL'] = channel_summary['Total_Spend'] / channel_summary['Total_Leads']
channel_summary['Overall_ROI'] = (channel_summary['Total_Revenue'] - channel_summary['Total_Spend']) / channel_summary['Total_Spend']

# Display the summary table sorted by ROI
channel_summary.sort_values(by='Overall_ROI', ascending=False)


,Channel,Total_Spend,Total_Revenue,Total_Clicks,Total_Impressions,Total_Leads,Overall_CTR,Overall_CPC,Cost_Per_Lead_CPL,Overall_ROI
1,google ads,364527.93,1232637.91,580409,12295858,19949.0,0.047204,0.628054,18.272993,2.381464
0,facebook,262581.24,887404.68,416567,8697799,13556.0,0.047893,0.630346,19.370112,2.379543
2,linkedin,246006.21,808543.89,400982,8363301,12293.0,0.047945,0.613509,20.011894,2.286681


In [ ]:
cleaned_df.to_csv('final_marketing_dashboard_data.csv', index=False)